# 05 — Evaluate

Reports detection metrics on the **LettuceMOTS val** split and on **your own frames** as **two separate tables** — never a single merged accuracy across public and own data.

Both are reported **per class**. A mean mAP over an imbalanced two-class set is close to meaningless here: with `healthy` outnumbering `unhealthy` many times over, a model that never predicts `unhealthy` at all still posts a respectable mean. The `unhealthy` column is the one that says whether the thing works.

The `labels` column records provenance: `colour-derived` for LettuceMOTS (the classes came from the colour rule, so the metric measures agreement with that rule) and `annotated` for your own frames (real labels, so the metric means what it usually means).

Needs the torch + ultralytics stack and a trained checkpoint.

In [ ]:
import os, sys
from pathlib import Path

# Locate repo root (holds croprow_disease/utils.py) so the package imports
# regardless of the cwd the notebook is launched from.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "croprow_disease" / "utils.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))
from croprow_disease import utils as U
from croprow_disease.health import HealthParams

CW = REPO_ROOT / "croprow_disease"
DATA_DIR = CW / "data"
MODELS_DIR = CW / "models"
RUNS_DIR = CW / "runs"
RESULTS_MD = CW / "RESULTS.md"

# ===================== CONFIG (edit here only) =====================
WEIGHTS       = str(MODELS_DIR / "best.pt")            # or best_finetuned.pt
HEALTH_YAML   = str(DATA_DIR / "health.yaml")          # from 01_dataset_prep
OWN_DATA_YAML = os.environ.get("OWN_DATA_YAML", "")    # "" = skip own-frames eval
IMGSZ         = 640
DEVICE        = 0
# ===================================================================
print("weights:", WEIGHTS)

## Environment check

In [ ]:
# This notebook needs the training/inference stack (torch + ultralytics),
# NOT installed in the light 01/02 env. See croprow_disease/requirements-train.txt.
try:
    import torch
    from ultralytics import YOLO
    import ultralytics
    print("torch      :", torch.__version__)
    print("ultralytics:", ultralytics.__version__)
    print("CUDA avail :", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"Missing training dependency: {e.name}. Install "
        "croprow_disease/requirements-train.txt into the croprow Python 3.11 venv "
        "before running this notebook."
    ) from e

In [ ]:
if not Path(WEIGHTS).is_file():
    raise FileNotFoundError(f"Weights not found: {WEIGHTS}. Train first (03/04).")

def report(tag, data_yaml):
    m = YOLO(WEIGHTS).val(data=data_yaml, imgsz=IMGSZ, device=DEVICE, split="val")
    per_class = {U.CLASS_NAMES[i]: float(v) for i, v in enumerate(m.box.maps)
                 if i < len(U.CLASS_NAMES)}
    row = dict(map50=float(m.box.map50), map5095=float(m.box.map),
               precision=float(m.box.mp), recall=float(m.box.mr),
               per_class=per_class)
    print(f"\n=== {tag} ===")
    print(f"  mAP50    : {row['map50']:.4f}")
    print(f"  mAP50-95 : {row['map5095']:.4f}")
    print(f"  precision: {row['precision']:.4f}")
    print(f"  recall   : {row['recall']:.4f}")
    for name, v in per_class.items():
        print(f"    {name:10s} mAP50-95: {v:.4f}")
    if per_class.get("unhealthy", 0.0) == 0.0:
        print("  NOTE: unhealthy mAP is 0 -- the model found none. Check the "
              "val split actually contains unhealthy instances before reading "
              "anything else in this table.")
    return row

## A) LettuceMOTS val (public data, colour-derived classes)

In [ ]:
pub = report("LettuceMOTS-val", HEALTH_YAML)
U.append_results_row(RESULTS_MD, run=Path(WEIGHTS).stem + "-eval",
                     dataset="LettuceMOTS-val", labels="colour-derived",
                     map50=pub["map50"], map5095=pub["map5095"],
                     precision=pub["precision"], recall=pub["recall"],
                     map50_healthy=pub["per_class"].get("healthy", "-"),
                     map50_unhealthy=pub["per_class"].get("unhealthy", "-"),
                     epochs="-", imgsz=IMGSZ)

## B) Your own frames (reported SEPARATELY)

Runs only if `OWN_DATA_YAML` is set. Logged as its own row with `labels="annotated"` — the public and own-data numbers are never combined.

In [ ]:
if OWN_DATA_YAML and Path(OWN_DATA_YAML).is_file():
    own = report("own-frames", OWN_DATA_YAML)
    U.append_results_row(RESULTS_MD, run=Path(WEIGHTS).stem + "-eval",
                         dataset="own-frames", labels="annotated",
                         map50=own["map50"], map5095=own["map5095"],
                         precision=own["precision"], recall=own["recall"],
                         map50_healthy=own["per_class"].get("healthy", "-"),
                         map50_unhealthy=own["per_class"].get("unhealthy", "-"),
                         epochs="-", imgsz=IMGSZ)
else:
    print("OWN_DATA_YAML not set / not found -> skipping own-frames eval "
          "(this is expected until you supply your frames).")

## RESULTS.md

In [ ]:
print(RESULTS_MD.read_text())